# 🧪 Evaluación de Modelos de Lenguaje con MLflow

---

## ¿De qué trata este cuaderno?

En este ejercicio aprenderás a usar el módulo de **evaluación de LLMs de MLflow** (`mlflow.genai.evaluate`),  
una herramienta diseñada para medir de forma sistemática la calidad de las respuestas que genera un modelo de lenguaje.

Al finalizar este cuaderno serás capaz de:

- ✅ Conectar tu código a un servidor MLflow remoto
- ✅ Diseñar un dataset de evaluación con preguntas y respuestas esperadas
- ✅ Aplicar evaluadores predefinidos (`Correctness`, `Guidelines`)
- ✅ Crear tu propio evaluador personalizado con `@scorer`
- ✅ Analizar los resultados en la interfaz web de MLflow

---

## 🗺️ Flujo general del experimento

```
Dataset de preguntas
        │
        ▼
  Función predict_fn  ──►  LLM (Claude / GPT / etc.)
        │
        ▼
   Respuestas generadas
        │
        ▼
    Scorers / Evaluadores
     ├── Correctness   (¿es correcta?)
     ├── Guidelines    (¿cumple reglas?)
     └── is_concise    (¿es concisa?)
        │
        ▼
   MLflow Tracking → UI web
```

---

> 📌 **Nota:** En este ejercicio usaremos la API de **Anthropic (Claude)** como modelo de lenguaje.  
> Si prefieres usar OpenAI u otro proveedor, los cambios son mínimos y se indican en cada celda.

---
# 📦 Sección 1: Instalación de dependencias

Antes de comenzar, instalaremos las librerías necesarias.

| Librería | ¿Para qué sirve? |
|---|---|
| `mlflow` | Tracking y evaluación de experimentos |
| `anthropic` | Cliente oficial de la API de Claude |
| `openai` | Alternativa si usas GPT |

> ⚠️ Si ya tienes estas librerías instaladas en tu entorno, puedes omitir esta celda.

In [1]:
# Instalar dependencias
#%pip install mlflow anthropic --quiet

# Si prefieres usar OpenAI en lugar de Anthropic:
%pip install mlflow openai --quiet

Note: you may need to restart the kernel to use updated packages.


---
# ⚙️ Sección 2: Configuración del entorno

## 2.1 ¿Qué es el Tracking URI?

El **Tracking URI** le dice a MLflow dónde guardar los resultados del experimento.  
Puede ser:
- Un servidor local (`http://127.0.0.1:5000`)
- Un servidor remoto como el que configuramos en clases anteriores
- Una carpeta local (`mlruns/` por defecto)

En este ejercicio usaremos el servidor remoto del curso.

## 2.2 API Key del modelo de lenguaje

Necesitas una clave de API del proveedor que vayas a usar.  
**Nunca escribas tu API key directamente en el código** — usa variables de entorno.

> 🔑 Para obtener una API key de Anthropic: https://console.anthropic.com  
> 🔑 Para obtener una API key de OpenAI: https://platform.openai.com

In [2]:
import os
import mlflow
import urllib3

# Silenciar advertencias de SSL (solo en entornos de desarrollo)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'

# ── Configuración de MLflow ──────────────────────────────────
MLFLOW_URI = "https://unab-n8n.duckdns.org/mlflow"  # Cambia por tu servidor
mlflow.set_tracking_uri(MLFLOW_URI)

# ── API Key del LLM ──────────────────────────────────────────
# Opción A: Anthropic (Claude)
#os.environ["ANTHROPIC_API_KEY"] = ""  # ← Reemplaza con tu key

# Opción B: OpenAI (descomenta si usas GPT)
os.environ["OPENAI_API_KEY"] = "tu-api-key-aqui"
# ── Verificar conexión ───────────────────────────────────────
print(f"✅ MLflow conectado a: {mlflow.get_tracking_uri()}")
print(f"📋 MLflow versión: {mlflow.__version__}")

✅ MLflow conectado a: https://unab-n8n.duckdns.org/mlflow
📋 MLflow versión: 3.11.1


---
# 🤖 Sección 3: Definir el agente (función predict)

## ¿Qué es la `predict_fn`?

La función de predicción (`predict_fn`) es el **corazón del experimento de evaluación**.  
MLflow la llama con cada pregunta del dataset y espera recibir una respuesta de texto.

```
predict_fn("¿Cuál es la capital de Francia?")  →  "París"
predict_fn("¿Quién escribió Don Quijote?")      →  "Miguel de Cervantes"
```

Puedes implementar esta función con **cualquier LLM**: Claude, GPT, Gemini, Llama, etc.  
Lo importante es que reciba un `str` y devuelva un `str`.

---

## 3.2 Alternativa con OpenAI (GPT)

Vamos a usar GPT

In [3]:

from openai import OpenAI

client = OpenAI()

def my_agent(question: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Eres un asistente útil. Responde de forma concisa."},
            {"role": "user",   "content": question},
        ]
    )
    return response.choices[0].message.content

def qa_predict_fn(question: str) -> str:
    return my_agent(question)


# ── Prueba rápida ────────────────────────────────────────────
respuesta_prueba = qa_predict_fn("¿Cuál es la capital de Colombia?")
print(f"🧪 Prueba del agente: {respuesta_prueba}")

🧪 Prueba del agente: La capital de Colombia es Bogotá.



## 3.2 Implementación con Claude (Anthropic)

Usaremos el modelo `claude-3-haiku-20240307` que es rápido y económico para pruebas.

```python

import anthropic

# Inicializar cliente de Anthropic
client = anthropic.Anthropic()

def my_agent(question: str) -> str:
    """
    Llama a Claude para responder una pregunta.
    
    Args:
        question: La pregunta en texto plano
    Returns:
        La respuesta generada por el modelo
    """
    message = client.messages.create(
        model="claude-3-haiku-20240307",   # Modelo rápido y económico
        max_tokens=256,
        system="Eres un asistente útil. Responde de forma concisa y precisa.",
        messages=[
            {"role": "user", "content": question}
        ]
    )
    return message.content[0].text


def qa_predict_fn(question: str) -> str:
    """
    Wrapper que MLflow usará para llamar al agente durante la evaluación.
    El nombre del parámetro ('question') debe coincidir con la clave
    del campo 'inputs' en el dataset de evaluación.
    """
    return my_agent(question)

```

---
# 📊 Sección 4: Dataset de evaluación

## ¿Qué es un dataset de evaluación?

Es un conjunto de pares **pregunta → respuesta esperada** que usamos como referencia  
para medir si el modelo está respondiendo correctamente.

Cada muestra tiene dos campos obligatorios:

| Campo | Descripción | Ejemplo |
|---|---|---|
| `inputs` | Lo que le enviamos al modelo | `{"question": "¿Capital de Francia?"}` |
| `expectations` | La respuesta correcta que esperamos | `{"expected_response": "París"}` |

> 💡 **La clave dentro de `inputs`** (`question`) debe coincidir exactamente  
> con el nombre del parámetro de tu función `predict_fn`.

---

## Diseño del dataset para este ejercicio

Usaremos preguntas de **geografía, historia y ciencia** con respuestas cortas y verificables.  
Esto facilita evaluar si el modelo responde correctamente y de forma concisa.

In [4]:
# Dataset de evaluación: preguntas con respuestas esperadas
eval_dataset = [
    # ── Geografía ─────────────────────────────────────────────
    {
        "inputs":       {"question": "¿Cuál es la capital de Francia?"},
        "expectations": {"expected_response": "París"},
    },
    {
        "inputs":       {"question": "¿Cuál es el río más largo del mundo?"},
        "expectations": {"expected_response": "El Nilo o el Amazonas"},
    },
    {
        "inputs":       {"question": "¿Cuántos continentes tiene la Tierra?"},
        "expectations": {"expected_response": "7"},
    },
    # ── Historia ──────────────────────────────────────────────
    {
        "inputs":       {"question": "¿Quién escribió Don Quijote de la Mancha?"},
        "expectations": {"expected_response": "Miguel de Cervantes"},
    },
    {
        "inputs":       {"question": "¿En qué año llegó el hombre a la Luna?"},
        "expectations": {"expected_response": "1969"},
    },
    # ── Ciencia ───────────────────────────────────────────────
    {
        "inputs":       {"question": "¿Cuál es la fórmula química del agua?"},
        "expectations": {"expected_response": "H2O"},
    },
    {
        "inputs":       {"question": "¿Cuántos planetas tiene el sistema solar?"},
        "expectations": {"expected_response": "8"},
    },
    {
        "inputs":       {"question": "¿Qué científica ganó dos premios Nobel en disciplinas distintas?"},
        "expectations": {"expected_response": "Marie Curie"},
    },
]

print(f"📋 Dataset cargado con {len(eval_dataset)} muestras")
print("\nEjemplo de muestra:")
import json
print(json.dumps(eval_dataset[0], indent=2, ensure_ascii=False))

📋 Dataset cargado con 8 muestras

Ejemplo de muestra:
{
  "inputs": {
    "question": "¿Cuál es la capital de Francia?"
  },
  "expectations": {
    "expected_response": "París"
  }
}


---
# 🎯 Sección 5: Definir los evaluadores (Scorers)

## ¿Qué es un Scorer?

Un **Scorer** es una función que recibe la pregunta, la respuesta del modelo y la respuesta esperada,  
y devuelve una **puntuación o un veredicto** sobre la calidad de esa respuesta.

MLflow incluye evaluadores predefinidos y permite crear los propios:

```
                    ┌─────────────────────────────────┐
                    │         SCORERS                 │
                    │                                 │
  Respuesta del  ──►│  Correctness  → ¿es correcta?  │
  modelo           │  Guidelines   → ¿cumple reglas?│
                    │  is_concise   → ¿es concisa?   │
                    │  is_spanish   → ¿en español?   │
                    └─────────────────────────────────┘
                                   │
                                   ▼
                           Puntuaciones → MLflow
```

---

## 5.1 Evaluadores predefinidos de MLflow

| Evaluador | Tipo | ¿Qué mide? |
|---|---|---|
| `Correctness()` | LLM-as-judge | Si la respuesta es objetivamente correcta |
| `Guidelines(...)` | LLM-as-judge | Si la respuesta cumple una regla en lenguaje natural |

Los evaluadores **LLM-as-judge** usan otro modelo de lenguaje para juzgar la calidad de la respuesta.  
Esto imita la evaluación humana y es mucho más escalable.

## 5.2 Evaluadores personalizados con `@scorer`

Puedes definir cualquier lógica de evaluación con Python puro usando el decorador `@scorer`.

In [5]:
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines


# ── Evaluador personalizado 1: Concisión ─────────────────────
@scorer
def is_concise(outputs: str) -> bool:
    """
    Evalúa si la respuesta es concisa (menos de 15 palabras).
    Devuelve True/False — MLflow lo convierte en 1.0 / 0.0.
    """
    return len(outputs.split()) <= 15


# ── Evaluador personalizado 2: Idioma español ─────────────────
@scorer
def is_spanish(outputs: str) -> bool:
    """
    Heurística simple para detectar si la respuesta está en español.
    Busca artículos o preposiciones comunes del español.
    """
    spanish_markers = [" el ", " la ", " los ", " las ", " de ", 
                       " en ", " es ", " son ", " del ", " con "]
    text_lower = outputs.lower()
    return any(marker in text_lower for marker in spanish_markers)


# ── Evaluador personalizado 3: No contiene disculpas ─────────
@scorer  
def no_apologies(outputs: str) -> bool:
    """
    Verifica que el modelo no empiece con disculpas innecesarias.
    Un buen asistente responde directamente.
    """
    apology_phrases = ["lo siento", "disculpa", "perdón", 
                       "i'm sorry", "i apologize"]
    return not any(phrase in outputs.lower() for phrase in apology_phrases)


# ── Lista final de evaluadores ────────────────────────────────
scorers = [
    Correctness(),      # ¿La respuesta es correcta? (LLM-as-judge)
    Guidelines(
        name="respuesta_en_espanol",
        guidelines="La respuesta debe estar escrita en español."
    ),
    Guidelines(
        name="sin_relleno",
        guidelines="La respuesta debe ir al grano, sin frases introductorias como 'claro', 'por supuesto' o 'ciertamente'."
    ),
    is_concise,         # ¿Menos de 15 palabras?
    is_spanish,         # ¿Detectamos español?
    no_apologies,       # ¿Sin disculpas innecesarias?
]

print(f"✅ {len(scorers)} evaluadores configurados:")
for s in scorers:
    print(f"   - {s.name if hasattr(s, 'name') else type(s).__name__}")

✅ 6 evaluadores configurados:
   - correctness
   - respuesta_en_espanol
   - sin_relleno
   - is_concise
   - is_spanish
   - no_apologies


---
# 🚀 Sección 6: Ejecutar la evaluación

## ¿Qué pasa cuando ejecutas `mlflow.genai.evaluate`?

MLflow realiza los siguientes pasos automáticamente:

1. **Itera** sobre cada muestra del dataset
2. **Llama** a tu `predict_fn` con la pregunta
3. **Aplica** cada scorer a la respuesta obtenida
4. **Registra** todos los resultados en el servidor de tracking
5. **Devuelve** un DataFrame con todos los resultados

```
Dataset (8 preguntas) × Scorers (6 evaluadores) = 48 evaluaciones individuales
```

> ⏱️ **Tiempo estimado:** ~1-2 minutos dependiendo de la velocidad de la API.

In [6]:
# Ejecutar la evaluación completa
print("🔄 Iniciando evaluación... esto puede tomar 1-2 minutos.")
print(f"   Preguntas a evaluar: {len(eval_dataset)}")
print(f"   Evaluadores activos: {len(scorers)}")
print()

mlflow.set_experiment("evaluacion-llm-preguntas-generales")

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=qa_predict_fn,
    scorers=scorers,
)

print("\n✅ Evaluación completada.")
print(f"📊 Revisa los resultados en: {mlflow.get_tracking_uri()}")

🔄 Iniciando evaluación... esto puede tomar 1-2 minutos.
   Preguntas a evaluar: 8
   Evaluadores activos: 6



2026/04/14 14:47:55 INFO mlflow.tracking.fluent: Experiment with name 'evaluacion-llm-preguntas-generales' does not exist. Creating a new experiment.
2026/04/14 14:47:55 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/14 14:47:55 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/8 [Elapsed: 00:00, Remaining: ?]


✅ Evaluación completada.
📊 Revisa los resultados en: https://unab-n8n.duckdns.org/mlflow


---
# 📈 Sección 7: Analizar los resultados

## 7.1 Resultados en Python

`mlflow.genai.evaluate` devuelve un objeto con los resultados completos.  
Puedes analizarlos directamente en el notebook antes de ir a la UI.

In [7]:
import pandas as pd

# Convertir resultados a DataFrame
df = results.tables["eval_results_table"]

print("=" * 60)
print("RESUMEN DE EVALUACIÓN")
print("=" * 60)

# Columnas disponibles
print(f"\n📋 Columnas disponibles: {list(df.columns)}\n")

# Mostrar tabla completa
display(df)

print("\n" + "=" * 60)
print("PROMEDIO POR EVALUADOR")
print("=" * 60)

# Calcular promedios de los scorers numéricos
scorer_cols = [c for c in df.columns 
               if c not in ["inputs", "outputs", "expectations"]]
for col in scorer_cols:
    try:
        mean_val = pd.to_numeric(df[col], errors='coerce').mean()
        pct = mean_val * 100 if mean_val <= 1 else mean_val
        bar = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"  {col:<30} {bar}  {pct:.0f}%")
    except Exception:
        pass

KeyError: 'eval_results_table'

## 7.2 Visualización en la UI de MLflow

La interfaz web de MLflow ofrece vistas mucho más ricas para analizar los resultados.

### Pasos para navegar la UI:

**1. Abrir el experimento**
- Ve a `https://unab-n8n.duckdns.org/mlflow/`
- Haz clic en **"evaluacion-llm-preguntas-generales"**

**2. Ver la tabla de resultados**
- Cada fila es una pregunta del dataset
- Las columnas muestran la puntuación de cada evaluador
- Las filas en rojo son las que fallaron algún evaluador

**3. Ver el detalle de una evaluación**
- Haz clic en cualquier fila
- Verás la pregunta, la respuesta del modelo y la justificación del evaluador LLM

**4. Comparar con una segunda ejecución**
- Ejecuta la celda de evaluación de nuevo (cambia el system prompt del agente)
- En la UI ve a "Evaluation Runs" → selecciona ambas ejecuciones → "Compare"

---

> 📌 **Dato clave:** El evaluador `Correctness` no solo dice si la respuesta es correcta —  
> también genera una **justificación en texto** explicando por qué la puntuó así.

---
# 🔬 Sección 8: Experimento comparativo

## ¿Qué pasa si cambiamos el system prompt?

Una de las ventajas de MLflow es poder comparar cómo cambia el desempeño  
del modelo al modificar el **system prompt** (las instrucciones del sistema).

En esta sección evaluaremos el mismo dataset con **dos versiones del agente**:
- **Versión A** — instrucciones genéricas (ya ejecutada arriba)
- **Versión B** — instrucciones que fuerzan respuestas de una sola palabra

Después compararemos los resultados en la UI de MLflow.

In [ ]:
# Versión B del agente: respuestas ultra-cortas
def my_agent_v2(question: str) -> str:
    """Agente con system prompt que fuerza respuestas de una sola palabra o número."""
    message = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=64,
        system=(
            "Eres un asistente de respuestas ultra-cortas. "
            "Responde SIEMPRE con una sola palabra o número. "
            "Sin explicaciones, sin puntos, sin artículos."
        ),
        messages=[{"role": "user", "content": question}]
    )
    return message.content[0].text

def qa_predict_fn_v2(question: str) -> str:
    return my_agent_v2(question)


# Ejecutar la evaluación con la versión B
print("🔄 Evaluando Versión B (respuestas ultra-cortas)...")

mlflow.set_experiment("evaluacion-llm-preguntas-generales")

results_v2 = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=qa_predict_fn_v2,
    scorers=scorers,
)

print("\n✅ Versión B evaluada.")
print("💡 Ahora ve a la UI de MLflow y compara ambas ejecuciones.")
print(f"   URL: {mlflow.get_tracking_uri()}")

---
# 📝 Sección 9: Ejercicios propuestos

Ahora que tienes el flujo completo funcionando, te proponemos los siguientes ejercicios:

---

### 🏋️ Ejercicio 1 — Ampliar el dataset
Agrega 5 preguntas nuevas al `eval_dataset` sobre un tema de tu elección  
(tecnología, literatura, matemáticas...) con sus respuestas esperadas.  
¿Cambia el accuracy promedio del modelo?

---

### 🏋️ Ejercicio 2 — Crear un scorer personalizado
Crea un evaluador llamado `is_numeric` que devuelva `True`  
si la respuesta contiene al menos un número.  
¿Cuántas preguntas del dataset esperan una respuesta numérica?

```python
@scorer
def is_numeric(outputs: str) -> bool:
    # Tu código aquí
    pass
```

---

### 🏋️ Ejercicio 3 — Comparar dos modelos
Modifica `my_agent` para usar `claude-3-opus-20240229` (modelo más potente)  
y vuelve a ejecutar la evaluación.  
Compara en la UI: ¿vale la pena el modelo más grande para estas preguntas?

---

### 🏋️ Ejercicio 4 — Guideline personalizada
Añade un evaluador `Guidelines` que verifique que las respuestas  
no incluyan información sobre eventos posteriores al año 2020.  
¿Qué preguntas del dataset podrían generar respuestas problemáticas?

---

### 🏋️ Ejercicio 5 — Preguntas trampa
Agrega al dataset 3 preguntas con premisas falsas, por ejemplo:  
*"¿En qué año llegó el hombre a Marte?"*  
Diseña un scorer que detecte si el modelo corrigió la premisa incorrecta  
en lugar de responder la pregunta tal cual.

---
# 🎓 Resumen

En este cuaderno aprendiste a:

| Concepto | Lo que hiciste |
|---|---|
| `predict_fn` | Envolviste un LLM (Claude) en una función evaluable |
| Dataset de evaluación | Creaste pares pregunta/respuesta esperada |
| `Correctness()` | Usaste un LLM como juez para verificar corrección |
| `Guidelines()` | Definiste reglas en lenguaje natural |
| `@scorer` | Escribiste evaluadores con lógica Python propia |
| `mlflow.genai.evaluate` | Ejecutaste la evaluación y la registraste en MLflow |
| UI de MLflow | Comparaste dos versiones del agente visualmente |

---

## 🔗 Recursos adicionales

- [Documentación oficial MLflow Evaluation](https://mlflow.org/docs/latest/genai/eval)
- [API de Anthropic](https://docs.anthropic.com)
- [Scorers disponibles en MLflow](https://mlflow.org/docs/latest/genai/eval/scorers)
- [LLM-as-Judge: artículo original](https://arxiv.org/abs/2306.05685)

---

> 🚀 **Siguiente paso:** En el próximo módulo veremos cómo registrar modelos en el  
> **Model Registry** de MLflow y hacer deployment a producción.